In [16]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx
import random
import time

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils.config import load_config
from src.utils.graph import radius_graph_pbc_batch
from src.utils.data import ot_alignment
from src.pbc_config import min_image, wrap, BOX
from src.validation import get_tpcf
from src.dataset import create_dataloader

In [2]:
device = torch.device("cpu")

In [21]:
train_dl, val_dl, _ = create_dataloader(
    "../../Data",
    "../../Data",
    False,
    train_kwargs={"batch_size": 4},
    valid_kwargs={"batch_size": 4},
)

In [22]:
sample = next(iter(train_dl))

In [5]:
x1 = sample.x
x0 = torch.rand(4*5000, 3)
batch = torch.arange(4, dtype=int).repeat_interleave(5000)
t = torch.rand(4, 1)
t = t[batch]
x_t = min_image(x1-x0, **BOX) * t + x0
x_t = x_t.view(4, 5000, 3)

for i in range(x_t.shape[0]):
    graph = x_t[i]
    edge_index = radius_graph_pbc_batch(graph, 0.25, torch.zeros(5000, dtype=int), device)

    G = nx.Graph()
    G.add_edges_from(edge_index.T.tolist())
    sample_nodes = random.sample(list(G.nodes()), 50)
    approx_diameter = max(nx.eccentricity(G, v=sample_nodes).values())
    print(f"Approximate diameter: {approx_diameter}")

RuntimeError: The size of tensor a (5000) must match the size of tensor b (50000) at non-singleton dimension 0

In [13]:
x1 = sample.x
x0 = torch.rand(5000, 3)
batch = torch.arange(1, dtype=int).repeat_interleave(5000)
t = torch.rand(1, 1)
t = t[batch]
x_t = min_image(x1-x0, **BOX) * t + x0
x_t = x_t.view(5000, 3)
edge_index = radius_graph_pbc_batch(x_t, 0.2, torch.zeros(5000, dtype=int), device)
print(f"Edges: {edge_index.shape[1]}")


Edges: 837240


In [24]:
start = time.time()

x1 = sample.x
x0 = torch.rand(4*5000, 3)
x0 = ot_alignment(x0, x1, 4)
batch = torch.arange(4, dtype=int).repeat_interleave(5000)
t = torch.rand(4, 1)
t = t[batch]
x_t = min_image(x1-x0, **BOX) * t + x0
edge_index = radius_graph_pbc_batch(x_t, 0.2, batch, device)
# x_t = x_t.view(4, 5000, 3)


end = time.time()

print(f"Total time for ot alignment: {end-start}")

Total time for ot alignment: 19.15744948387146
